# Probability Through Problem Solving

Notebook for **Chapter 1** of the *stats* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/stats/ch01.html

Tip: change a number, predict the output, then run the cell again.

## 1.1 Counting and Sample Spaces

In [ ]:
from math import comb, perm
print(comb(10, 3), perm(10, 3), comb(9, 2) / comb(10, 3))

120 720 0.3


In [ ]:
import numpy as np
def p_match(n):
    p = 1.0
    for i in range(n):
        p *= (365 - i) / 365
    return 1 - p
print("exact n=23:", round(p_match(23), 4), " n=50:", round(p_match(50), 4))
rng = np.random.default_rng(1)
teams = rng.integers(0, 365, size=(100_000, 23))
hits = sum(len(set(row)) < 23 for row in teams)
print("simulated n=23:", hits / 100_000)

exact n=23: 0.5073  n=50: 0.9704
simulated n=23: 0.50695


In [ ]:
# plot
import numpy as np, matplotlib.pyplot as plt
ns = np.arange(1, 81)
ps = [p_match(int(k)) for k in ns]
plt.figure(figsize=(6, 3.6))
plt.plot(ns, ps, color="#1f77b4", lw=2)
plt.axhline(0.5, color="gray", ls="--")
plt.axvline(23, color="gray", ls=":")
plt.xlabel("People in the team"); plt.ylabel("P(at least two share a birthday)")
plt.title("Birthday problem: 23 people give about 50 percent")
plt.tight_layout(); plt.savefig(f"{IMG}/ch01_birthday.png", dpi=130); plt.close()

## 1.2 Conditional Probability and Independence

In [ ]:
import numpy as np
rng = np.random.default_rng(2)
older = rng.integers(0, 2, 1_000_000)   # 1 = girl
younger = rng.integers(0, 2, 1_000_000)
atleast = (older == 1) | (younger == 1)
both = (older == 1) & (younger == 1)
print("P(both | at least one girl):", round(both[atleast].mean(), 4))
print("P(both | older is girl):   ", round(both[older == 1].mean(), 4))

P(both | at least one girl): 0.3344
P(both | older is girl):    0.5009


## 1.3 Bayes' Theorem and the Base-Rate Trap

In [ ]:
def posterior(prior, sens, fpr):
    pe = sens * prior + fpr * (1 - prior)
    return sens * prior / pe
print("posterior:", round(posterior(0.005, 0.99, 0.02), 4))
for prior in [0.0005, 0.005, 0.05, 0.2, 0.5]:
    print(f"prior={prior:<7} posterior={posterior(prior, 0.99, 0.02):.4f}")

posterior: 0.1992
prior=0.0005  posterior=0.0242
prior=0.005   posterior=0.1992
prior=0.05    posterior=0.7226
prior=0.2     posterior=0.9252
prior=0.5     posterior=0.9802


In [ ]:
prior_odds = 0.005 / 0.995
lr = 0.99 / 0.02
odds = prior_odds * lr**2
print("posterior after two alerts:", round(odds / (1 + odds), 4))

posterior after two alerts: 0.9249


In [ ]:
import numpy as np
rng = np.random.default_rng(3)
n = 200_000
car = rng.integers(0, 3, n)
pick = np.zeros(n, dtype=int)            # always pick door 0
stay = (car == pick).mean()
switch = (car != pick).mean()
print("stay:", round(stay, 3), " switch:", round(switch, 3))

stay: 0.334  switch: 0.666


## 1.4 Random Variables, Expectation and Variance

In [ ]:
import numpy as np
v = np.array([100, 300, 1000]); p = np.array([0.6, 0.3, 0.1])
m = (v * p).sum(); var = (v**2 * p).sum() - m**2
print("mean", m, "var", var, "sd", round(var**0.5, 2))
rng = np.random.default_rng(4)
fails = rng.binomial(1, 0.03, size=(500_000, 20)).sum(axis=1)
print("fails mean/var:", round(fails.mean(), 3), round(fails.var(), 3))
print("P(at least one failure):", round(1 - 0.97**20, 4), " simulated:", round((fails > 0).mean(), 4))
n_people = 10
matches = [(rng.permutation(n_people) == np.arange(n_people)).sum() for _ in range(100_000)]
print("expected own hats:", round(np.mean(matches), 3))

mean 250.0 var 70500.0 sd 265.52
fails mean/var: 0.597 0.58
P(at least one failure): 0.4562  simulated: 0.4544
expected own hats: 1.003


## 1.5 Descriptive Statistics and Correlation Done Properly

In [ ]:
import numpy as np
from scipy import stats
rng = np.random.default_rng(5)
delivery_minutes = rng.lognormal(mean=5.3, sigma=0.5, size=1000)   # median about 200 minutes
def describe(x):
    return {"mean": x.mean(), "median": np.median(x), "sd": x.std(ddof=1),
            "MAD": stats.median_abs_deviation(x), "IQR": stats.iqr(x),
            "skew": stats.skew(x), "kurt": stats.kurtosis(x),
            "p95": np.percentile(x, 95), "p99": np.percentile(x, 99)}
a = describe(delivery_minutes)
b = describe(np.concatenate([delivery_minutes, np.full(5, 20000.0)]))
print(f"{'stat':8}{'clean':>12}{'with outliers':>16}")
for k in a:
    print(f"{k:8}{a[k]:12.2f}{b[k]:16.2f}")

stat           clean   with outliers
mean          226.76          325.13
median        202.32          203.26
sd            119.06         1396.97
MAD            67.14           68.10
IQR           141.81          143.14
skew            1.63           13.92
kurt            4.56          193.14
p95           461.16          475.15
p99           590.08          681.91


In [ ]:
x = np.arange(1, 21, dtype=float)
y = x**3
print("cubic   pearson %.3f spearman %.3f kendall %.3f" % (
    stats.pearsonr(x, y)[0], stats.spearmanr(x, y)[0], stats.kendalltau(x, y)[0]))
y2 = 2 * x; y2[-1] = -200
print("outlier pearson %.3f spearman %.3f kendall %.3f" % (
    stats.pearsonr(x, y2)[0], stats.spearmanr(x, y2)[0], stats.kendalltau(x, y2)[0]))

cubic   pearson 0.922 spearman 1.000 kendall 1.000
outlier pearson -0.168 spearman 0.714 kendall 0.800


## 1.6 Simpson's Paradox and Why Correlation Is Not Causation

In [ ]:
import pandas as pd
d = pd.DataFrame({"build": ["A","A","B","B"], "traffic": ["light","heavy","light","heavy"],
                  "pass": [81, 192, 234, 55], "n": [87, 263, 270, 80]})
d["rate"] = (d["pass"] / d["n"]).round(3)
print(d)
tot = d.groupby("build")[["pass", "n"]].sum()
tot["rate"] = (tot["pass"] / tot["n"]).round(3)
print(tot)

  build traffic  pass    n   rate
0     A   light    81   87  0.931
1     A   heavy   192  263  0.730
2     B   light   234  270  0.867
3     B   heavy    55   80  0.688
       pass    n   rate
build                  
A       273  350  0.780
B       289  350  0.826


## Chapter recap

## Practice set

Problem 1.7.1 How many different 5-letter passwords are possible using 26 lowercase letters with no repeated letter?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/stats/ch01.html